# 🧱 Part 12: LoRA: Low-Rank Adaptation

> **Previous context**: We have a trainable model, but full fine-tuning can be expensive.
> **Goal for this part**: Implement LoRA, understand low-rank updates, and see how adapters can be merged for inference.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Why LoRA exists

A pretrained model already stores broad knowledge. For a new task, we often only need a small update rather than changing every weight.

## 1. Low-rank idea

Instead of learning a full weight update matrix, LoRA learns two smaller matrices A and B whose product approximates the update.

## 2. Where to apply it

LoRA is commonly attached to attention projections or other linear layers where small targeted changes are useful.

## 3. Merge for inference

After training, the low-rank update can be folded into the base weight so inference stays simple.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] LoRA freezes the base weight and learns a low-rank update.
- [ ] A and B are much smaller than a full update matrix.
- [ ] LoRA can often be merged into the base model for inference.

Next, continue through the code cells for the Training Systems part and inspect the printed observations.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

In [ ]:
# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
torch.manual_seed(42)

d_in, d_out, r = 4, 4, 2
alpha = 2  # Teaching note: follow this line to see the main step.

# Teaching note: follow this line to see the main step.
W = torch.randn(d_out, d_in)
print('Read the values printed above and connect them to the concept in this cell.')
print(W)
print('Read the values printed above and connect them to the concept in this cell.')

# Teaching note: follow this line to see the main step.
A = torch.randn(d_out, r) * 0.02   # Teaching note: follow this line to see the main step.
B = torch.zeros(r, d_in)            # Teaching note: follow this line to see the main step.
print(f"\nA (d_out x r = 4x2):")
print(A)
print('Read the values printed above and connect them to the concept in this cell.')
print(B)
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

# Teaching note: follow this line to see the main step.
x = torch.tensor([1.0, 0.5, -0.5, -1.0])
print('f"\\nInput x (4,): {x}"')

# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
# Teaching note: follow this line to see the main step.
h_original = W @ x
print('Read the values printed above and connect them to the concept in this cell.')

# Teaching note: follow this line to see the main step.
h_lora = B @ x     # Teaching note: follow this line to see the main step.
h_lora = A @ h_lora  # Teaching note: follow this line to see the main step.
scaling = alpha / r
delta = h_lora * scaling
print('Read the values printed above and connect them to the concept in this cell.')
print(f"  Step 1 - Bx (r={r},): {h_lora}")
print(f"  Step 2 - A(Bx) (d_out={d_out},): {h_lora}")
print('Read the values printed above and connect them to the concept in this cell.')
print(f"  delta = (alpha/r) * A(Bx): {delta}")

# Teaching note: follow this line to see the main step.
h_final = h_original + delta
print('Read the values printed above and connect them to the concept in this cell.')

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print('Training')

In [ ]:
# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
class LoraLinear(nn.Module):
    'Training'
    def __init__(self, linear: nn.Linear, r: int, alpha: int = None):
        super().__init__()
        # Teaching note: follow this line to see the main step.
        self.linear = linear
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.r = r
        self.alpha = alpha if alpha is not None else r

        # Teaching note: follow this line to see the main step.
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

        # Teaching note: follow this line to see the main step.
        # A: (out_features, r)  B: (r, in_features)
        self.lora_A = nn.Parameter(torch.randn(self.out_features, r) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, self.in_features))
        self.scaling = self.alpha / self.r

    def forward(self, x):
        # Teaching note: follow this line to see the main step.
        original = self.linear(x)  # x @ W^T + bias
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        lora_out = (x @ self.lora_B.T) @ self.lora_A.T * self.scaling
        return original + lora_out

    @property
    def weight(self):
        'Read the values printed above and connect them to the concept in this cell.'
        with torch.no_grad():
            delta_W = self.lora_A @ self.lora_B * self.scaling
            return self.linear.weight + delta_W


# Teaching note: follow this line to see the main step.
torch.manual_seed(42)
linear = nn.Linear(4, 4, bias=False)
linear.weight.data = torch.randn(4, 4)  # Teaching note: follow this line to see the main step.

lora_layer = LoraLinear(linear, r=2, alpha=2)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print(f"  lora_A requires_grad: {lora_layer.lora_A.requires_grad}")
print(f"  lora_B requires_grad: {lora_layer.lora_B.requires_grad}")

# Teaching note: follow this line to see the main step.
total = sum(p.numel() for p in lora_layer.parameters())
trainable = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
print('Training')

# Teaching note: follow this line to see the main step.
x = torch.tensor([[1.0, 0.5, -0.5, -1.0]])
y = lora_layer(x)
print('f"\\n  Input x: {x.tolist()}"')
print('f"  Output y: {y.tolist()}"')

In [ ]:
# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
torch.manual_seed(42)

# Teaching note: follow this line to see the main step.
d = 4
W_target = torch.tensor([
    [ 0.5, -0.3,  0.8, -0.2],
    [-0.4,  0.6, -0.1,  0.7],
    [ 0.3, -0.5, -0.6,  0.4],
    [-0.7,  0.2,  0.5, -0.8],
])

N = 30
X_train = torch.randn(N, d)
y_train = X_train @ W_target + 0.05 * torch.randn(N, d)  # Teaching note: follow this line to see the main step.

print('Training')
print('Read the values printed above and connect them to the concept in this cell.')

# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
torch.manual_seed(100)
W_full = nn.Linear(d, d, bias=False)
W_full.weight.data = torch.randn(d, d)  # Teaching note: follow this line to see the main step.

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Training')

opt_full = torch.optim.Adam(W_full.parameters(), lr=0.01)
losses_full = []
for epoch in range(200):
    opt_full.zero_grad()
    loss = F.mse_loss(W_full(X_train), y_train)
    loss.backward()
    opt_full.step()
    losses_full.append(loss.item())

print('Loss')

# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
torch.manual_seed(100)  # Teaching note: follow this line to see the main step.
W_base = nn.Linear(d, d, bias=False)
W_base.weight.data = torch.randn(d, d)  # Teaching note: follow this line to see the main step.

# Teaching note: follow this line to see the main step.
lora_model = LoraLinear(W_base, r=2, alpha=4)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Training')

opt_lora = torch.optim.Adam(lora_model.parameters(), lr=0.01)
losses_lora = []
for epoch in range(200):
    opt_lora.zero_grad()
    loss = F.mse_loss(lora_model(X_train), y_train)
    loss.backward()
    opt_lora.step()
    losses_lora.append(loss.item())

print('Loss')

# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
print('Read the values printed above and connect them to the concept in this cell.')
print('Loss')
print('Loss')

# Teaching note: follow this line to see the main step.
with torch.no_grad():
    learned_W = lora_model.weight  # Teaching note: follow this line to see the main step.
    print('Read the values printed above and connect them to the concept in this cell.')
    print('Read the values printed above and connect them to the concept in this cell.')
    print('Loss')

    full_W = W_full.weight.data
    print('Loss')

In [ ]:
# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
def get_sinusoidal_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        Q = self.W_Q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.W_O(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, max_len=128):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.register_buffer('pos_encoding', get_sinusoidal_encoding(max_len, d_model))
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.norm_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, mask=None):
        B, S = x.shape
        x_emb = self.token_embedding(x) + self.pos_encoding[:S, :]
        for block in self.blocks:
            x_emb = block(x_emb, mask)
        x_emb = self.norm_final(x_emb)
        return self.lm_head(x_emb)

# Teaching note: follow this line to see the main step.
def causal_mask(seq_len):
    return torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)

print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
def apply_lora_to_attention(model, r=16, alpha=32):
    'Read the values printed above and connect them to the concept in this cell.'
    lora_params = 0
    for block in model.blocks:
        attn = block.attention
        # Teaching note: follow this line to see the main step.
        attn.W_Q = LoraLinear(attn.W_Q, r=r, alpha=alpha)
        # Teaching note: follow this line to see the main step.
        attn.W_V = LoraLinear(attn.W_V, r=r, alpha=alpha)
        # Teaching note: follow this line to see the main step.
        lora_params += attn.W_Q.lora_A.numel() + attn.W_Q.lora_B.numel()
        lora_params += attn.W_V.lora_A.numel() + attn.W_V.lora_B.numel()
    return lora_params

VOCAB_SIZE = 50
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)

# Teaching note: follow this line to see the main step.
total_params = sum(p.numel() for p in model.parameters())
print('Read the values printed above and connect them to the concept in this cell.')

# Teaching note: follow this line to see the main step.
lora_params = apply_lora_to_attention(model, r=8, alpha=16)

# Teaching note: follow this line to see the main step.
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Training')
print('Read the values printed above and connect them to the concept in this cell.')

# Teaching note: follow this line to see the main step.
for name, param in model.named_parameters():
    if 'lora' not in name and 'lm_head' not in name and 'token_embedding' not in name:
        if param.requires_grad:
            print(f"  WARNING: {name} should be frozen but requires_grad=True")
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
word_to_id = {
    "BOS": 0, "EOS": 1,
    '"hello"': 2, '"thanks"': 3, '"goodbye"': 4, '"today"': 5, '"weather"': 6, '"nice"': 7,
    '"you"': 8, '"is"': 9, 'Read the values printed above and connect them to the concept in this cell.': 10, '"I"': 11, '"assistant"': 12, "AI": 13,
    'Read the values printed above and connect them to the concept in this cell.': 14, 'Read the values printed above and connect them to the concept in this cell.': 15, 'Read the values printed above and connect them to the concept in this cell.': 16, 'Read the values printed above and connect them to the concept in this cell.': 17, '"can"': 18,
    '"good"': 19, '"bad"': 20, '"like"': 21, '"dislike"': 22, '"know"': 23, '"do not know"': 24,
    '"math"': 25, '"English"': 26, '"programming"': 27, 'Read the values printed above and connect them to the concept in this cell.': 28, '"code"': 29,
    '"today"': 5, '"weekday"': 30, 'Read the values printed above and connect them to the concept in this cell.': 31, 'Read the values printed above and connect them to the concept in this cell.': 32, 'Read the values printed above and connect them to the concept in this cell.': 33,
}

# Teaching note: follow this line to see the main step.
word_to_id = {
    "BOS": 0, "EOS": 1, "PAD": 0,
    '"hello"': 2, '"thanks"': 3, '"goodbye"': 4, '"today"': 5, '"weather"': 6, '"nice"': 7,
    '"you"': 8, '"is"': 9, 'Read the values printed above and connect them to the concept in this cell.': 10, '"I"': 11, '"assistant"': 12, "AI": 13,
    'Read the values printed above and connect them to the concept in this cell.': 14, 'Read the values printed above and connect them to the concept in this cell.': 15, 'Read the values printed above and connect them to the concept in this cell.': 16, 'Read the values printed above and connect them to the concept in this cell.': 17, '"can"': 18,
    '"good"': 19, '"bad"': 20, '"like"': 21, '"dislike"': 22, '"know"': 23, '"do not know"': 24,
    '"math"': 25, '"English"': 26, '"programming"': 27, 'Read the values printed above and connect them to the concept in this cell.': 28, '"code"': 29,
    '"weekday"': 30, 'Read the values printed above and connect them to the concept in this cell.': 31, 'Read the values printed above and connect them to the concept in this cell.': 32, 'Read the values printed above and connect them to the concept in this cell.': 33, 'Read the values printed above and connect them to the concept in this cell.': 34,
    'Read the values printed above and connect them to the concept in this cell.': 35, 'Read the values printed above and connect them to the concept in this cell.': 36, 'Read the values printed above and connect them to the concept in this cell.': 37, 'Read the values printed above and connect them to the concept in this cell.': 38, 'Read the values printed above and connect them to the concept in this cell.': 39,
    'Read the values printed above and connect them to the concept in this cell.': 40, 'Read the values printed above and connect them to the concept in this cell.': 41, 'Read the values printed above and connect them to the concept in this cell.': 42, 'Read the values printed above and connect them to the concept in this cell.': 43, 'Read the values printed above and connect them to the concept in this cell.': 44,
    'Read the values printed above and connect them to the concept in this cell.': 45, 'Read the values printed above and connect them to the concept in this cell.': 46, 'Read the values printed above and connect them to the concept in this cell.': 47, 'Read the values printed above and connect them to the concept in this cell.': 48, 'Read the values printed above and connect them to the concept in this cell.': 49,
}

# Teaching note: follow this line to see the main step.
conversations = [
    ('"hello"', 'Read the values printed above and connect them to the concept in this cell.'),
    ('"todayweathernice"', 'Read the values printed above and connect them to the concept in this cell.'),
    ('Read the values printed above and connect them to the concept in this cell.', 'Read the values printed above and connect them to the concept in this cell.'),
    ('Read the values printed above and connect them to the concept in this cell.', 'Read the values printed above and connect them to the concept in this cell.'),
    ('"thanks"', 'Read the values printed above and connect them to the concept in this cell.'),
    ('"goodbye"', 'Read the values printed above and connect them to the concept in this cell.'),
]

def encode_text(text):
    'Read the values printed above and connect them to the concept in this cell.'
    ids = []
    for word in text:
        if word in word_to_id:
            ids.append(word_to_id[word])
        else:
            ids.append(0)  # Teaching note: follow this line to see the main step.
    return ids

print('Training')
for q, a in conversations[:3]:
    print(f"  User: {q}  →  Assistant: {a}")
    q_ids = encode_text(q)
    a_ids = encode_text(a)
    print(f"    IDs: User {q_ids}, Assistant {a_ids}")

In [ ]:
# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
VOCAB_SIZE = 50
SEQ_LEN = 24
torch.manual_seed(42)

# Teaching note: follow this line to see the main step.
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2, max_len=SEQ_LEN)
apply_lora_to_attention(model, r=8, alpha=16)

# Teaching note: follow this line to see the main step.
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.001
)

print('Read the values printed above and connect them to the concept in this cell.')

# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
NUM_EPOCHS = 50
losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0

    for user_text, assistant_text in conversations:
        # Teaching note: follow this line to see the main step.
        user_ids = encode_text(user_text)
        asst_ids = encode_text(assistant_text)
        full_seq = [word_to_id["BOS"]] + user_ids + asst_ids + [word_to_id["EOS"]]

        # Teaching note: follow this line to see the main step.
        if len(full_seq) > SEQ_LEN:
            full_seq = full_seq[:SEQ_LEN]
        input_ids = torch.tensor([full_seq + [0] * (SEQ_LEN - len(full_seq))])

        # Teaching note: follow this line to see the main step.
        mask = causal_mask(SEQ_LEN)
        logits = model(input_ids, mask)  # (1, S, V)

        # Teaching note: follow this line to see the main step.
        labels = torch.cat([input_ids[:, 1:], torch.zeros(1, 1, dtype=torch.long)], dim=1)

        # Teaching note: follow this line to see the main step.
        loss = F.cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            labels.view(-1),
            ignore_index=0  # PAD
        )

        epoch_loss += loss.item()

        # Teaching note: follow this line to see the main step.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = epoch_loss / len(conversations)
    losses.append(avg_loss)

    if epoch % 5 == 0 or epoch == NUM_EPOCHS - 1:
        print(f"Epoch {epoch:3d} | Loss: {avg_loss:.4f}")

print('Loss')
print('Loss')

In [ ]:
# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(losses, 'b-', linewidth=1.5)
plt.axhline(y=losses[0], color='gray', linestyle='--', alpha=0.5, label=f'Initial: {losses[0]:.2f}')
plt.axhline(y=losses[-1], color='red', linestyle='--', alpha=0.5, label=f'Final: {losses[-1]:.2f}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LoRA fine-tuning MiniGPT — loss curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Conclusion: this small example shows the main trade-off.')
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# ============================================================
# Teaching note: follow this line to see the main step.
# ============================================================
def merge_lora(model):
    'Read the values printed above and connect them to the concept in this cell.'
    for block in model.blocks:
        attn = block.attention
        for name in ['W_Q', 'W_V']:
            layer = getattr(attn, name)
            if isinstance(layer, LoraLinear):
                # Teaching note: follow this line to see the main step.
                merged_weight = layer.linear.weight.data + layer.lora_A.data @ layer.lora_B.data * layer.scaling
                # Teaching note: follow this line to see the main step.
                new_linear = nn.Linear(layer.in_features, layer.out_features, bias=False)
                new_linear.weight.data = merged_weight
                setattr(attn, name, new_linear)

# Teaching note: follow this line to see the main step.
trainable_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Training')

merge_lora(model)

trainable_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Training')
print('Read the values printed above and connect them to the concept in this cell.')